In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Feature engineering libraries imported successfully!")

Feature engineering libraries imported successfully!


In [2]:
from pathlib import Path

DATA_PATH = Path("../data/processed/career_dataset_normalized.csv")

career_df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", career_df.shape)
display(career_df.head())

Dataset loaded successfully!
Dataset shape: (9000, 16)


,Education Level,Specialization,Skills,Certifications,CGPA/Percentage,Recommended Career,Certification_Status,Score_Type,Score_Value,Technical_Skills_Parsed,Soft_Skills_Parsed,Skills_Gained_Parsed,Technical_Skill_Count,Soft_Skill_Count,Skills_Gained_Count,Recommended_Career_Original
0,BA,Agriculture,Technical: Microsoft Excel; Python; Data analy...,Not specified,8.22,M.Sc Biotechnology / Microbiology / Life Sciences,Not specified,CGPA,8.22,Microsoft Excel; Python; Data analysis; Biolog...,Critical thinking; Problem-solving,Research methods; Data analysis; Biology funda...,4,2,5,M.Sc Biotechnology / Microbiology / Life Sciences
1,Class 10,Not yet selected,Technical: Digital creativity; Visual storytel...,Introductory online course,56.00,Arts / Humanities or a relevant design-oriente...,Provided,Percentage,56.00,Digital creativity; Visual storytelling,Research; Communication,Design fundamentals; Drawing,2,2,2,Arts / Humanities or a relevant design-oriente...
2,Class 10,Not yet selected,Technical: Subject knowledge; Presentation | S...,Introductory online course,84.50,Choose a suitable stream based on preferred te...,Provided,Percentage,84.50,Subject knowledge; Presentation,Patience; Empathy; Adaptability,Subject knowledge; Communication,2,3,2,Choose a suitable stream based on preferred te...
3,Class 12,CEC,Technical: Planning; Excel basics | Soft: Rese...,Basic computer course,40.80,B.Com / BBA / B.A. Economics,Provided,Percentage,40.80,Planning; Excel basics,Research; Teamwork; Presentation; Leadership,Planning; Presentation; Business communication,2,4,3,B.Com / BBA / B.A. Economics
4,B.Des,Pharmacy,Technical: Environmental awareness; Python; Ob...,SWAYAM certificate,7.27,M.Sc Agriculture / Horticulture / Food Science...,Provided,CGPA,7.27,Environmental awareness; Python; Observation; ...,Research; Presentation; Adaptability; Leadership,Data recording; Data analysis; Environmental a...,5,4,4,M.Sc Agriculture / Horticulture / Food Science...


In [3]:
# Define input features
feature_columns = [
    "Education Level",
    "Specialization",
    "Certifications",
    "Score_Type",
    "Score_Value",
    "Technical_Skills_Parsed",
    "Soft_Skills_Parsed",
    "Skills_Gained_Parsed"
]

# Separate features and target
X = career_df[feature_columns]
y = career_df["Recommended Career"]

print("Input features shape:", X.shape)
print("Target shape:", y.shape)

print("\nInput feature columns:")
print(X.columns.tolist())

print("\nNumber of career classes:", y.nunique())

Input features shape: (9000, 8)
Target shape: (9000,)

Input feature columns:
['Education Level', 'Specialization', 'Certifications', 'Score_Type', 'Score_Value', 'Technical_Skills_Parsed', 'Soft_Skills_Parsed', 'Skills_Gained_Parsed']

Number of career classes: 30


In [4]:
categorical_features = [
    "Education Level",
    "Specialization",
    "Certifications",
    "Score_Type"
]

numerical_features = [
    "Score_Value"
]

text_features = [
    "Technical_Skills_Parsed",
    "Soft_Skills_Parsed",
    "Skills_Gained_Parsed"
]

print("Categorical features:", categorical_features)
print("\nNumerical features:", numerical_features)
print("\nText features:", text_features)

print("\nFeature count:", 
      len(categorical_features) + 
      len(numerical_features) + 
      len(text_features))

Categorical features: ['Education Level', 'Specialization', 'Certifications', 'Score_Type']

Numerical features: ['Score_Value']

Text features: ['Technical_Skills_Parsed', 'Soft_Skills_Parsed', 'Skills_Gained_Parsed']

Feature count: 8


In [5]:
# Initialize One-Hot Encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit and transform categorical features
encoded_data = encoder.fit_transform(
    X[categorical_features]
)

print("Original categorical shape:", X[categorical_features].shape)
print("Encoded categorical shape:", encoded_data.shape)

print("\nFirst 5 encoded records:")
display(pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_features)
).head())

Original categorical shape: (9000, 4)
Encoded categorical shape: (9000, 39)

First 5 encoded records:


,Education Level_B.Ag,Education Level_B.Com,Education Level_B.Des,Education Level_B.Pharm,Education Level_B.Sc,Education Level_B.Tech,Education Level_BA,Education Level_BBA,Education Level_BCA,Education Level_Class 10,...,Certifications_Industry-oriented certificate,Certifications_Introductory online course,Certifications_NPTEL course certificate,Certifications_Not specified,Certifications_Online introductory certificate,Certifications_SWAYAM certificate,Certifications_School-level STEM certificate,Certifications_Subject-related certificate,Score_Type_CGPA,Score_Type_Percentage
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [6]:
# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform numerical features
scaled_data = scaler.fit_transform(
    X[numerical_features]
)

scaled_df = pd.DataFrame(
    scaled_data,
    columns=numerical_features
)

print("Original Score_Value statistics:")
display(X[numerical_features].describe())

print("\nScaled Score_Value statistics:")
display(scaled_df.describe())

Original Score_Value statistics:


,Score_Value
count,9000.000000
mean,53.756621
std,33.924660
min,4.340000
25%,8.200000
50%,69.200000
75%,80.700000
max,99.500000



Scaled Score_Value statistics:


,Score_Value
count,9.000000e+03
mean,6.947529e-17
std,1.000056e+00
min,-1.456739e+00
25%,-1.342951e+00
50%,4.552510e-01
75%,7.942563e-01
max,1.348456e+00


In [7]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(ngram_range=(1, 2))

# Transform technical skills into numerical features
technical_tfidf = tfidf.fit_transform(
    X["Technical_Skills_Parsed"]
)

print("Original technical skill records:", X["Technical_Skills_Parsed"].shape[0])
print("TF-IDF matrix shape:", technical_tfidf.shape)

print("\nSample vocabulary:")
print(list(tfidf.vocabulary_.keys())[:20])

Original technical skill records: 9000
TF-IDF matrix shape: (9000, 443)

Sample vocabulary:
['microsoft', 'excel', 'python', 'data', 'analysis', 'biology', 'fundamentals', 'microsoft excel', 'excel python', 'python data', 'data analysis', 'analysis biology', 'biology fundamentals', 'digital', 'creativity', 'visual', 'storytelling', 'digital creativity', 'creativity visual', 'visual storytelling']


In [8]:
# Initialize separate TF-IDF vectorizers
soft_tfidf = TfidfVectorizer(ngram_range=(1, 2))
gained_tfidf = TfidfVectorizer(ngram_range=(1, 2))

# Transform soft skills
soft_tfidf_matrix = soft_tfidf.fit_transform(
    X["Soft_Skills_Parsed"]
)

# Transform skills gained
gained_tfidf_matrix = gained_tfidf.fit_transform(
    X["Skills_Gained_Parsed"]
)

print("Soft Skills TF-IDF shape:", soft_tfidf_matrix.shape)
print("Skills Gained TF-IDF shape:", gained_tfidf_matrix.shape)

print("\nSample Soft Skills vocabulary:")
print(list(soft_tfidf.vocabulary_.keys())[:15])

print("\nSample Skills Gained vocabulary:")
print(list(gained_tfidf.vocabulary_.keys())[:15])

Soft Skills TF-IDF shape: (9000, 215)
Skills Gained TF-IDF shape: (9000, 356)

Sample Soft Skills vocabulary:
['critical', 'thinking', 'problem', 'solving', 'critical thinking', 'thinking problem', 'problem solving', 'research', 'communication', 'research communication', 'patience', 'empathy', 'adaptability', 'patience empathy', 'empathy adaptability']

Sample Skills Gained vocabulary:
['research', 'methods', 'data', 'analysis', 'biology', 'fundamentals', 'observation', 'chemistry', 'research methods', 'methods data', 'data analysis', 'analysis biology', 'biology fundamentals', 'fundamentals observation', 'observation chemistry']


In [9]:
# Build the complete preprocessing transformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "technical_skills",
            TfidfVectorizer(ngram_range=(1, 2)),
            "Technical_Skills_Parsed"
        ),
        (
            "soft_skills",
            TfidfVectorizer(ngram_range=(1, 2)),
            "Soft_Skills_Parsed"
        ),
        (
            "skills_gained",
            TfidfVectorizer(ngram_range=(1, 2)),
            "Skills_Gained_Parsed"
        )
    ],
    remainder="drop"
)

print("ColumnTransformer created successfully!")

ColumnTransformer created successfully!


In [10]:
# Apply preprocessing to the dataset
combined_features = preprocessor.fit_transform(X)

print("Original dataset shape:", X.shape)
print("Combined feature matrix shape:", combined_features.shape)

print("\nIs the feature matrix sparse?")
print(hasattr(combined_features, "toarray"))

print("\nTotal transformed features:", combined_features.shape[1])

Original dataset shape: (9000, 8)
Combined feature matrix shape: (9000, 1054)

Is the feature matrix sparse?
True

Total transformed features: 1054


In [11]:
from sklearn.model_selection import train_test_split

# Split input features and target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nNumber of career classes in training:", y_train.nunique())
print("Number of career classes in testing:", y_test.nunique())

Training features shape: (7200, 8)
Testing features shape: (1800, 8)

Training target shape: (7200,)
Testing target shape: (1800,)

Number of career classes in training: 30
Number of career classes in testing: 30


In [12]:
# Create a fresh preprocessor for model training

model_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "technical_skills",
            TfidfVectorizer(ngram_range=(1, 2)),
            "Technical_Skills_Parsed"
        ),
        (
            "soft_skills",
            TfidfVectorizer(ngram_range=(1, 2)),
            "Soft_Skills_Parsed"
        ),
        (
            "skills_gained",
            TfidfVectorizer(ngram_range=(1, 2)),
            "Skills_Gained_Parsed"
        )
    ],
    remainder="drop"
)

# Fit only on training data
X_train_transformed = model_preprocessor.fit_transform(X_train)

# Transform test data without fitting again
X_test_transformed = model_preprocessor.transform(X_test)

print("Training transformed shape:", X_train_transformed.shape)
print("Testing transformed shape:", X_test_transformed.shape)

print("\nPreprocessing fitted on training data only!")

Training transformed shape: (7200, 1054)
Testing transformed shape: (1800, 1054)

Preprocessing fitted on training data only!


In [13]:
print("CAREERPATH FEATURE ENGINEERING SUMMARY")
print("-" * 45)

print("Original records:", len(career_df))
print("Original input features:", X.shape[1])
print("Target classes:", y.nunique())

print("\nTrain-test split:")
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTransformed features:")
print("Training features:", X_train_transformed.shape[1])
print("Testing features:", X_test_transformed.shape[1])

print("\nValidation checks:")
print("Training and testing feature counts match:",
      X_train_transformed.shape[1] == X_test_transformed.shape[1])

print("All career classes in training:",
      y_train.nunique() == y.nunique())

print("All career classes in testing:",
      y_test.nunique() == y.nunique())

print("\nFEATURE ENGINEERING COMPLETED!")

CAREERPATH FEATURE ENGINEERING SUMMARY
---------------------------------------------
Original records: 9000
Original input features: 8
Target classes: 30

Train-test split:
Training records: 7200
Testing records: 1800

Transformed features:
Training features: 1054
Testing features: 1054

Validation checks:
Training and testing feature counts match: True
All career classes in training: True
All career classes in testing: True

FEATURE ENGINEERING COMPLETED!


# CareerPath AI – Feature Engineering Summary

## 1. Dataset Overview
- Total records: 9,000
- Original input features: 8
- Target variable: `Recommended Career`
- Number of career classes: 30

## 2. Feature Engineering Techniques

### Categorical Features
Applied One-Hot Encoding to:
- Education Level
- Specialization
- Certifications
- Score_Type

### Numerical Features
Applied StandardScaler to:
- Score_Value

### Text Features
Applied TF-IDF Vectorization with unigrams and bigrams to:
- Technical_Skills_Parsed
- Soft_Skills_Parsed
- Skills_Gained_Parsed

## 3. Feature Transformation
- Original features: 8
- Transformed features: 1,054
- Combined feature matrix: Sparse matrix

## 4. Train-Test Split
Used stratified train-test splitting.

- Training records: 7,200 (80%)
- Testing records: 1,800 (20%)
- Random state: 42
- Career classes represented in both datasets: 30

## 5. Data Leakage Prevention
- Created a fresh preprocessing transformer for model training.
- Fitted preprocessing only on training data.
- Transformed test data without refitting.
- Ensured identical feature dimensions for training and testing.

## 6. Final Validation
All validation checks passed:
- Feature dimensions match.
- All 30 career classes are present in training data.
- All 30 career classes are present in testing data.
- Training and testing transformations completed successfully.

## Conclusion
Feature engineering successfully converted categorical, numerical, and text-based student profile information into 1,054 numerical features. The transformed data is ready for CareerPath AI model training and evaluation.

**Notebook Status: Completed**
